# Nationality Prediction from First and Last Names

This notebook uses the `name2nat` model (included in this repository) to predict the nationality origin of names from `data/firstname.csv` and `data/lastname.csv`.

Each output CSV retains the original columns and adds:
- `predicted_nationality` — the top-1 predicted nationality
- `confidence` — the model's confidence score (0–1)

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
from name2nat import Name2nat
from tqdm.auto import tqdm

In [ ]:
model = Name2nat()
print('Model loaded successfully.')

In [ ]:
def predict_nationalities(names, model, batch_size=1024):
    """Predict nationality for a list of names, processing in batches."""
    nationalities = [None] * len(names)
    confidences = [None] * len(names)

    valid_indices = [i for i, n in enumerate(names) if isinstance(n, str) and n.strip()]
    valid_names = [names[i] for i in valid_indices]
    print(f'  Valid names: {len(valid_names):,} / {len(names):,}')

    for i in tqdm(range(0, len(valid_names), batch_size), desc='Predicting'):
        batch = valid_names[i:i + batch_size]
        batch_indices = valid_indices[i:i + batch_size]
        results = model(batch, top_n=1, use_dict=True)
        for j, (_, preds) in enumerate(results):
            idx = batch_indices[j]
            if preds:
                nationalities[idx] = preds[0][0]
                confidences[idx] = round(preds[0][1], 6)
    return nationalities, confidences

## First Names

In [ ]:
df_first = pd.read_csv('data/firstname.csv')
print(f'Loaded {len(df_first):,} first names')
df_first.head()

In [ ]:
first_names = df_first['first_name'].tolist()
nat, conf = predict_nationalities(first_names, model)
df_first['predicted_nationality'] = nat
df_first['confidence'] = conf
df_first.head(10)

In [ ]:
df_first.to_csv('data/firstname_predicted.csv', index=False)
print(f'Saved data/firstname_predicted.csv ({len(df_first):,} rows)')

## Last Names

In [ ]:
df_last = pd.read_csv('data/lastname.csv')
print(f'Loaded {len(df_last):,} last names')
df_last.head()

In [ ]:
last_names = df_last['last_name'].tolist()
nat, conf = predict_nationalities(last_names, model)
df_last['predicted_nationality'] = nat
df_last['confidence'] = conf
df_last.head(10)

In [ ]:
df_last.to_csv('data/lastname_predicted.csv', index=False)
print(f'Saved data/lastname_predicted.csv ({len(df_last):,} rows)')

## Summary Statistics

In [ ]:
for label, df in [('First names', df_first), ('Last names', df_last)]:
    valid = df['confidence'].notna()
    print(f'\n=== {label} ===')
    print(f'Total names: {len(df):,}')
    print(f'Predicted: {valid.sum():,}')
    print(f'Mean confidence: {df["confidence"].mean():.4f}')
    print(f'Median confidence: {df["confidence"].median():.4f}')
    print(f'Names with confidence >= 0.9: {(df["confidence"] >= 0.9).sum():,} ({(df.loc[valid, "confidence"] >= 0.9).mean():.1%})')
    print(f'Names with confidence < 0.5: {(df["confidence"] < 0.5).sum():,} ({(df.loc[valid, "confidence"] < 0.5).mean():.1%})')
    print(f'\nTop 10 predicted nationalities:')
    print(df['predicted_nationality'].value_counts().head(10).to_string())